In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import re
import time
import json
import os

# ===============================
# 🛠️ SETUP SELENIUM DRIVER
# ===============================
def setup_driver():
    """Configures and returns a headless Selenium WebDriver."""
    options = Options()
    options.add_argument("--headless")  # Run without opening a window
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920x1080")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

# ===============================
# 📄 GET TOTAL NUMBER OF PAGES
# ===============================
def get_total_pages(driver, base_url):
    """Finds the total number of event pages from pagination UI."""
    driver.get(base_url + "1")
    time.sleep(3)  # Allow JavaScript to load

    pagination_links = driver.find_elements("css selector", "ul.pagination li a")
    page_numbers = [int(link.text.strip()) for link in pagination_links if link.text.strip().isdigit()]
    
    return max(page_numbers) if page_numbers else 1

# ===============================
# 📝 EXTRACT EVENT JSON TEXT
# ===============================
def extract_event_text(page_source):
    """Extracts the JSON event text from the page source using regex."""
    match = re.search(r'var\s+events\s*=\s*(\[.*?\]);', page_source, re.DOTALL)
    return match.group(1).strip() if match else None

# ===============================
# 🔄 SCRAPE EVENTS FOR ANY HOST
# ===============================
def scrape_events(event_host_id, event_host_name, check_for_new=False):
    """
    Scrapes all events for a given host from SmoothComp.
    
    :param event_host_id: ID of the event host (e.g., 176 for ADCC).
    :param event_host_name: Name of the event host for file naming.
    :param check_for_new: If True, scrape only new events that haven't been stored before.
    :return: List of extracted event data.
    """
    
    # Define base URL dynamically
    base_url = f"https://smoothcomp.com/en/federation/{event_host_id}/events/past?page="
    driver = setup_driver()
    total_pages = get_total_pages(driver, base_url)
    all_events = []

    # Load previously saved event IDs if checking for new events
    existing_event_ids = set()
    events_file = f"data/{event_host_name}_events.json"
    
    if check_for_new and os.path.exists(events_file):
        with open(events_file, "r", encoding="utf-8") as f:
            existing_event_ids = {event["id"] for event in json.load(f)}
        print(f"Loaded {len(existing_event_ids)} existing event IDs to avoid re-scraping.")

    # Scrape events page by page
    for page in range(1, total_pages + 1):
        print(f"Fetching events from page {page} of {total_pages}...")
        driver.get(base_url + str(page))
        time.sleep(3)

        event_text = extract_event_text(driver.page_source)
        if event_text:
            try:
                event_data = json.loads(event_text)  # Convert raw text to JSON
                new_events = [event for event in event_data if event["id"] not in existing_event_ids]
                all_events.extend(new_events)
            except json.JSONDecodeError:
                print(f"⚠️ Failed to parse JSON on page {page}")
    
    driver.quit()

    # Save results
    if all_events:
        os.makedirs("data", exist_ok=True)
        with open(events_file, "w", encoding="utf-8") as f:
            json.dump(all_events, f, indent=4)
        print(f"✅ Saved {len(all_events)} new events to {events_file}")

    return all_events

# ===============================
# 🏁 EXECUTE SCRAPER
# ===============================
if __name__ == "__main__":
    events = scrape_events(event_host_id=176, event_host_name="ADCC", check_for_new=True)
    print(f"Extracted {len(events)} new events.")


Fetching page 1 of 2...
Saved raw HTML for page 1 for debugging.
No events JSON found on page 1.
Fetching page 2 of 2...
Saved raw HTML for page 2 for debugging.
No events JSON found on page 2.
Successfully saved raw events text to all_events_raw.txt
Extracted event data saved to all_events_raw.txt
